In [3]:
import pandas as pd
matches = pd.read_csv("../data/raw/international_results.csv")

In [6]:
matches["date"] = pd.to_datetime(matches["date"])
matches = matches.sort_values("date")


In [10]:
elo = {}

def get_rating(team):
    if team not in elo:
        elo[team] = 1500
    return elo[team]

def expected_score(team_rating, opponent_rating):
    return 1 / (1 + 10 ** ((opponent_rating - team_rating) / 400))

def actual_score(team_goals, opponent_goals):
    if team_goals > opponent_goals:
        return 1

    if team_goals == opponent_goals:
        return 0.5

    return 0

def update_rating(old_rating, expected, actual, k=20):
    return old_rating + k * (actual - expected)


In [11]:
brazil_rating = 1500
argentina_rating = 1500

brazil_expected = expected_score(brazil_rating, argentina_rating)
argentina_expected = expected_score(argentina_rating, brazil_rating)

brazil_actual = actual_score(2, 1)
argentina_actual = actual_score(1, 2)

new_brazil_rating = update_rating(
    brazil_rating,
    brazil_expected,
    brazil_actual
)

new_argentina_rating = update_rating(
    argentina_rating,
    argentina_expected,
    argentina_actual
)

print(new_brazil_rating)
print(new_argentina_rating)

1510.0
1490.0


In [13]:
home_elo_before = []
away_elo_before = []
for _, match in matches.iterrows():
    home_team = match["home_team"]
    away_team = match["away_team"]

    home_rating = get_rating(home_team)
    away_rating = get_rating(away_team)

    home_elo_before.append(home_rating)
    away_elo_before.append(away_rating)

    home_expected = expected_score(home_rating, away_rating)
    away_expected = expected_score(away_rating, home_rating)

    home_actual = actual_score(match["home_score"], match["away_score"])
    away_actual = actual_score(match["away_score"], match["home_score"])

    elo[home_team] = update_rating(home_rating, home_expected, home_actual)
    elo[away_team] = update_rating(away_rating, away_expected, away_actual)

matches["home_elo_before"] = home_elo_before
matches["away_elo_before"] = away_elo_before
matches["elo_difference"] = (
    matches["home_elo_before"] - matches["away_elo_before"]
)